In [19]:
import mlflow
from mlflow.tracking import MlflowClient

# ==================== Check MLflow Status ====================
mlflow.set_tracking_uri('http://localhost:5000')

client = MlflowClient()

# Get experiment
experiment = client.get_experiment_by_name('predictive-maintenance')

if experiment:
    print(f"✅ Experiment Found!")
    print(f"Experiment Name : {experiment.name}")
    print(f"Experiment ID   : {experiment.experiment_id}")
    print(f"Artifact Location: {experiment.artifact_location}")

    # List all runs in this experiment
    runs = client.search_runs(experiment.experiment_id)
    print(f"\nTotal Runs Found: {len(runs)}")

    for run in runs:
        print(f"   - Run Name: {run.info.run_name} | Status: {run.info.status}")

else:
    print("❌ Experiment 'predictive-maintenance' not found!")

✅ Experiment Found!
Experiment Name : predictive-maintenance
Experiment ID   : 1
Artifact Location: file:C:/Users/PMLS/PythonProject4/mlruns/1

Total Runs Found: 29
   - Run Name: xgboost | Status: FINISHED
   - Run Name: random_forest | Status: FINISHED
   - Run Name: logistic_regression | Status: FINISHED
   - Run Name: xgboost | Status: FINISHED
   - Run Name: random_forest | Status: FINISHED
   - Run Name: logistic_regression | Status: FINISHED
   - Run Name: xgboost | Status: FINISHED
   - Run Name: random_forest | Status: FINISHED
   - Run Name: logistic_regression | Status: FINISHED
   - Run Name: xgboost | Status: FINISHED
   - Run Name: random_forest | Status: FINISHED
   - Run Name: logistic_regression | Status: FINISHED
   - Run Name: xgboost | Status: FINISHED
   - Run Name: random_forest | Status: FINISHED
   - Run Name: logistic_regression | Status: FINISHED
   - Run Name: xgboost | Status: FINISHED
   - Run Name: random_forest | Status: FINISHED
   - Run Name: logistic_r

In [20]:
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd

# ==================== Check MLflow Status ====================
mlflow.set_tracking_uri('http://localhost:5000')

client = MlflowClient()

# Get experiment
experiment = client.get_experiment_by_name('predictive-maintenance')

if experiment:
    print(f"✅ Experiment Found!")
    print(f"Experiment Name : {experiment.name}")
    print(f"Experiment ID   : {experiment.experiment_id}\n")

    # Get all runs sorted by ROC AUC (Best to Worst)
    runs = client.search_runs(
        experiment_ids=experiment.experiment_id,
        order_by=['metrics.roc_auc DESC']
    )

    print(f"Total Runs Found: {len(runs)}\n")

    # Prepare data for DataFrame
    run_data = []
    for run in runs:
        run_data.append({
            'run_id': run.info.run_id[:8],
            'name': run.info.run_name,
            'model_type': run.data.params.get('model_type', 'Unknown'),
            'roc_auc': round(run.data.metrics.get('roc_auc', 0), 4),
            'f1_score': round(run.data.metrics.get('f1_score', 0), 4),
            'accuracy': round(run.data.metrics.get('accuracy', 0), 4)
        })

    df = pd.DataFrame(run_data)
    print('All Runs (sorted by ROC AUC - Best First):')
    print(df)

else:
    print("❌ Experiment 'predictive-maintenance' not found!")

✅ Experiment Found!
Experiment Name : predictive-maintenance
Experiment ID   : 1

Total Runs Found: 29

All Runs (sorted by ROC AUC - Best First):
      run_id                 name          model_type  roc_auc  f1_score  \
0   dea1998a        random_forest        RandomForest   0.9753    0.6012   
1   5df66ff5        random_forest        RandomForest   0.9753    0.6012   
2   91c84f96        random_forest        RandomForest   0.9751    0.5976   
3   18a8807d        random_forest        RandomForest   0.9751    0.5976   
4   a63aacb2        random_forest        RandomForest   0.9751    0.5976   
5   8c50c502        random_forest        RandomForest   0.9751    0.5976   
6   06770266        random_forest        RandomForest   0.9751    0.5976   
7   3768d9bf        random_forest        RandomForest   0.9751    0.5976   
8   5451236c        random_forest        RandomForest   0.9751    0.5976   
9   aefb81dc              xgboost             XGBoost   0.9709    0.5926   
10  5b452816     

In [21]:
# Fetch metadata and metrics
version = registered_model.version
roc_auc = best_run.data.metrics['roc_auc']
f1 = best_run.data.metrics['f1_score']

# Define a detailed markdown description
description = f'''
### XGBoost Predictive Maintenance Model
* **Dataset:** 10,000 equipment sensor samples
* **Performance:** ROC AUC: {roc_auc:.4f} | F1 Score: {f1:.4f}
* **Features Used:** Temperature, vibration, pressure, RPM, age_days
'''

# Update MLflow Registry with documentation and tracking tags
client.update_model_version(name=model_name, version=version, description=description)
client.set_model_version_tag(model_name, version, 'validation_status', 'passed')
client.set_model_version_tag(model_name, version, 'team', 'datascience')
client.set_model_version_tag(model_name, version, 'framework', 'xgboost')

print('Documentation added!')

Documentation added!


In [23]:
# Transition the current model version to 'Staging' and archive older versions
client.transition_model_version_stage(
    name=model_name,
    version=version,
    stage='Staging',
    archive_existing_versions=True
)
print(f'Model version {version} moved to Staging')

# Load the staging model back to verify it works properly
staging_model = mlflow.pyfunc.load_model(
    model_uri=f'models:/{model_name}/Staging'
)
print('Staging model loaded successfully!')

C:\Users\PMLS\AppData\Local\Temp\ipykernel_104276\2846676452.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


Model version 1 moved to Staging
Staging model loaded successfully!


In [24]:
import numpy as np

# 1. Create a simulated high-risk test sample
test_data = pd.DataFrame({
     'temperature': [95],   # High
     'vibration': [0.9],   # High
     'pressure': [135],    # High
     'rpm': [1500],
     'age_days': [320]     # Old
})

# 2. Get prediction using the Staging model
prediction = staging_model.predict(test_data)

# 3. Print the results
print('Test Prediction (Staging Model):')
print(f'Input: High temp, high vibration, old equipment')
print(f'Prediction: {"FAILURE LIKELY" if prediction[0] == 1 else "NO FAILURE"}')
print(f'Raw Output: {prediction[0]}')

Test Prediction (Staging Model):
Input: High temp, high vibration, old equipment
Prediction: FAILURE LIKELY
Raw Output: 1


C:\Users\PMLS\PythonProject4\.venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


In [25]:
from datetime import datetime

# 1. Promote the approved model version to Production
client.transition_model_version_stage(
     name=model_name,
     version=version,
     stage='Production',
     archive_existing_versions=True
)
print(f'Model version {version} is now in PRODUCTION!')

# 2. Add a tag with today's deployment date
client.set_model_version_tag(
     model_name,
     version,
     'deployment_date',
     datetime.now().strftime('%Y-%m-%d')
)

C:\Users\PMLS\AppData\Local\Temp\ipykernel_104276\1180032955.py:4: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


Model version 1 is now in PRODUCTION!


In [28]:
import pandas as pd
import mlflow
import mlflow.pyfunc
import pickle
import os

# ===================================================
# TASK 3.2: PRODUCTION INFERENCE PIPELINE (DOOSRI FILE KE LIYE)
# ===================================================
def predict_equipment_failure(temperature, vibration, pressure, rpm, age_days):
    """
    MLflow Production stage se model load karta hai aur save kiye gaye
    scaler.pkl ko read karke inputs ko transform karta hai.
    """
    # 1. Production registry se live model load kiya
    model_uri = f"models:/PredictiveMaintenance/Production"
    model = mlflow.pyfunc.load_model(model_uri)

    # 2. Input values ka DataFrame banaya exact sequence mein
    input_df = pd.DataFrame([{
        'temperature': float(temperature),
        'vibration': float(vibration),
        'pressure': float(pressure),
        'rpm': float(rpm),
        'age_days': int(age_days)
    }])

    # 3. TRANSFORMATION FIX: Pehli file se save kiye gaye scaler ko load kiya
    if os.path.exists('scaler.pkl'):
        with open('scaler.pkl', 'rb') as f:
            local_scaler = pickle.load(f)
        input_data_scaled = local_scaler.transform(input_df)
    else:
        # Agar file nahi milti toh warning dega (Safety Check)
        print("❌ Error: 'scaler.pkl' file nahi mili! Pehli file mein scaler save karein.")
        return None

    # 4. Prediction generate ki
    prediction = model.predict(input_data_scaled)[0]

    return {
        'will_fail': bool(prediction),
        'recommendation': 'Schedule maintenance' if int(prediction) == 1 else 'Normal operation'
    }


# --- Testing 3 Scenarios with Real Numbers ---
scenarios = [
    {'name': 'Normal', 'temp': 70, 'vib': 0.4, 'press': 95, 'rpm': 1500, 'age': 100},
    {'name': 'High Risk', 'temp': 95, 'vib': 0.9, 'press': 135, 'rpm': 1500, 'age': 320},
    {'name': 'Medium', 'temp': 85, 'vib': 0.6, 'press': 110, 'rpm': 1500, 'age': 200}
]

print('Production Predictions (Via MLflow Server + Loaded Scaler):')
print('-' * 55)

for s in scenarios:
    result = predict_equipment_failure(
        s['temp'], s['vib'], s['press'], s['rpm'], s['age']
    )
    if result:
        print(f"{s['name']:12} → {result['recommendation']}")

print('-' * 55)

Production Predictions (Via MLflow Server + Loaded Scaler):
-------------------------------------------------------
Normal       → Normal operation
High Risk    → Schedule maintenance
Medium       → Normal operation
-------------------------------------------------------


In [29]:
import mlflow
import pickle
import os
import pandas as pd

# ===================================================
# TASK 4: REGISTER VERSION 2 & TEST ROLLBACK
# ===================================================

# 1. Doosra model register kiya (Simulate new version)
# Ensure karein 'runs' variable aur 'model_name' aapki is file mein pehle se defined hain
second_run = runs[1]
model_uri = f'runs:/{second_run.info.run_id}/model'

v2 = mlflow.register_model(model_uri, model_name)
print(f'Registered version {v2.version}')

# 2. Rollback: Move v1 back to Production stage
print('\nRolling back to version 1...')
client.transition_model_version_stage(
     name=model_name,
     version=1,
     stage='Production',
     archive_existing_versions=True
)
print('✓ Rollback complete!')
print('Production now uses version 1 again\n')


# ===================================================
# POST-ROLLBACK: LIVE INFERENCE TESTING WITH SCALER
# ===================================================
print('Testing Production Inference after Rollback:')
print('-' * 55)

# Safety Check: Pehli file se save kiye gaye scaler ko load karenge
if os.path.exists('scaler.pkl'):
    with open('scaler.pkl', 'rb') as f:
        production_scaler = pickle.load(f)

    # Production model ko load kiya (Jo rollback ke baad ab version 1 ban chuka hai)
    prod_model_uri = f"models:/{model_name}/Production"
    production_model = mlflow.pyfunc.load_model(prod_model_uri)

    # Hum usi functional scenarios ko test karte hain jo lab sheet mein hain
    scenarios = [
        {'name': 'Normal', 'temp': 70, 'vib': 0.4, 'press': 95, 'rpm': 1500, 'age': 100},
        {'name': 'High Risk', 'temp': 95, 'vib': 0.9, 'press': 135, 'rpm': 1500, 'age': 320}
    ]

    for s in scenarios:
        # DataFrame banaya input ka
        input_df = pd.DataFrame([{
            'temperature': float(s['temp']),
            'vibration': float(s['vib']),
            'pressure': float(s['press']),
            'rpm': float(s['rpm']),
            'age_days': int(s['age'])
        }])

        # Data ko transform/scale kiya taaki output sahi aaye
        input_data_scaled = production_scaler.transform(input_df)

        # Rollback hue model se prediction li
        prediction = production_model.predict(input_data_scaled)[0]
        recommendation = 'Schedule maintenance' if int(prediction) == 1 else 'Normal operation'

        print(f"{s['name']:12} → {recommendation}")

else:
    print("❌ Error: 'scaler.pkl' nahi mila! Pehli file (training file) mein scaler ko save karein.")

print('-' * 55)

Registered model 'PredictiveMaintenance' already exists. Creating a new version of this model...
2026/06/07 17:49:33 WARNING mlflow.tracking._model_registry.fluent: Run with id 5df66ff53620487c897d1137020e2497 has no artifacts at artifact path 'model', registering model based on models:/m-ce5cc05812c8423ca0f8cac9bf5f29ae instead
2026/06/07 17:49:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: PredictiveMaintenance, version 2
Created version '2' of model 'PredictiveMaintenance'.
C:\Users\PMLS\AppData\Local\Temp\ipykernel_104276\3343368624.py:20: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_

Registered version 2

Rolling back to version 1...
✓ Rollback complete!
Production now uses version 1 again

Testing Production Inference after Rollback:
-------------------------------------------------------
Normal       → Normal operation
High Risk    → Schedule maintenance
-------------------------------------------------------
